# Bolo 03: Whisper fine-tune ablation

Same data and code as notebook 02, no baselines. Variants:
- `freeze_enc`: encoder frozen, only the decoder trains (less forgetting of general acoustics?)
- `short`: full model, 2 epochs instead of 5
Each is scored on the same synthetic test (held-out voices). Output: `/kaggle/working/ablation/<variant>/` + `results_ablation.json`.

In [ ]:
import glob, os, sys, json
# Kaggle: bundle uploaded as a private dataset -> /kaggle/input/...; Colab: unzip bolo-bundle.zip into /content
cands = glob.glob("/kaggle/input/**/bolo-bundle/src/bolo/__init__.py", recursive=True) + \
        glob.glob("/content/**/bolo-bundle/src/bolo/__init__.py", recursive=True)
assert cands, "bolo-bundle not found: add the bolo-bundle dataset (Kaggle) or unzip it in /content (Colab)"
BUNDLE = cands[0].split("/src/bolo/")[0]
sys.path.insert(0, BUNDLE + "/src")
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
print("bundle:", BUNDLE, "| work dir:", WORK)

tm = glob.glob("/kaggle/input/**/bolo-tts/manifest.jsonl", recursive=True) + glob.glob(WORK + "/bolo-tts/manifest.jsonl")
assert tm, "add the bolo-tts-v1 dataset (or notebook 01's output) as an input"
TTS_MAN = tm[0]
print("tts manifest:", TTS_MAN)

In [ ]:
import torch, time
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from bolo.asr import whisper_ft as W
from bolo.config import load_config
from bolo.normalize import Normalizer
nz = Normalizer.from_config(load_config())
rows = W.load_manifest(TTS_MAN, for_training=True)
split = {s: [r for r in rows if r["split"] == s] for s in ("train", "dev", "test")}
split["train"] = split["train"] + W.nonspeech_rows(max(1, len(split["train"]) * 3 // 100))  # ~3% noise-only
print({k: len(v) for k, v in split.items()})
DEV = "cuda"
RESULTS = {"data": {k: len(v) for k, v in split.items()}, "baselines": {}, "finetuned": {}}

In [ ]:
VARIANTS = {"freeze_enc": dict(epochs=5, freeze_encoder=True, lr=1e-5),
            "short": dict(epochs=2, freeze_encoder=False, lr=1e-5)}
for name, v in VARIANTS.items():
    proc = WhisperProcessor.from_pretrained(W.BASE)
    model = WhisperForConditionalGeneration.from_pretrained(W.BASE)
    out = f"{WORK}/ablation/{name}"
    t0 = time.time()
    res = W.train(model, proc, split["train"], split["dev"], out, device=DEV, normalizer=nz, batch_size=32,
                  warmup=50, eval_every=150, seed=0, **v)
    best = WhisperForConditionalGeneration.from_pretrained(out, torch_dtype=torch.float16).to(DEV)
    s, _ = W.evaluate(best, WhisperProcessor.from_pretrained(out), split["test"], nz, language="en", device=DEV)
    RESULTS["finetuned"][name] = {**s, "config": v, "minutes": round((time.time() - t0) / 60, 1), "best_dev": res["best"]}
    print(name, RESULTS["finetuned"][name], flush=True)
    del model, best; torch.cuda.empty_cache()
json.dump(RESULTS, open(WORK + "/results_ablation.json", "w"), indent=1, ensure_ascii=False)